[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrew-l-miller/gwosc/blob/main/make_sfts/make_SFTs.ipynb)

In [ ]:
from pathlib import Path
import os
import sys
import requests
import argparse
import matplotlib.pyplot as plt
import numpy as np
import scipy
import glob
import shlex
import shutil
import subprocess
import re

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    
if IN_COLAB:
    if os.path.basename(os.getcwd()) == "make_sfts":
        print('things already downloaded')
    else:
        print("downloading codes and compiled code")
        !git clone https://github.com/andrew-l-miller/gwosc.git
        os.chdir('gwosc/make_sfts')
    
    
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt
!wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt



### Install environment to run make SFTs C code in

In [ ]:
if IN_COLAB:
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !./bin/micromamba create -y -n igwn -c conda-forge lalpulsar

#### Check to see that lalpulsar_MakeSFTs (the function to create SFTs) is installed

In [ ]:
!./bin/micromamba run -n igwn which lalpulsar_MakeSFTs

### Import functions to download gravitational-wave strain data from GWOSC and to create SFTs

In [ ]:
sys.path.append(os.path.abspath("../create_sfdbs"))
from download_all_data_from_run import download_gwf
from make_ffl import *
from utils import make_cache_file, make_sfts

### Choose which observing run, interferometer, start/end time to analyze, which channel, which freq band

If you want to download data for the whole run, simply put `gps_start=gps_end=None`

In [ ]:
obs_run = 'O4a'
ifo = 'H1'
runn = obs_run+'_4KHZ_R1'
channel = ifo+':GWOSC-4KHZ_R1_STRAIN'
gwf_dir = './data/'+obs_run+'/'+ifo+'/'
gps_start = 1369185055
gps_end = gps_start+10*4096
start_freq = 10.0
band = 1600.0
outdir = "./sfts/"+obs_run+'/'+ifo+'/'

### Download the desired gravitational-wave frame (.gwf) files

In [ ]:
download_gwf(runn,ifo,gwf_dir,gps_start,gps_end)

### Make a list of GW frame files and save it as an .ffl file

In [ ]:
ffl_name = ifo+'_'+channel+'.ffl'
make_ffl(gwf_dir,ffl_name,absolute=True)

### Create the .cache file (a necessary input for making SFTs)

In [ ]:
cache_fname = ifo+'_'+obs_run+'.cache'
make_cache_file(ffl=ffl_name,ifo=ifo,channel_id=channel,cache_out=cache_fname)


### MAKE SFTs

In [ ]:
make_sfts(
    cache_file=cache_fname,
    gps_start_time=gps_start,
    gps_end_time=gps_end,
    sft_write_path=outdir,
    start_freq=start_freq,
    band=band,
    channel_name=channel
)


### Making sanity-check plots

In [ ]:
!pip install pyfstat
import pyfstat
from scipy.signal import medfilt


In [ ]:
freqs, times, sft_data = pyfstat.utils.get_sft_as_arrays(outdir+'*.sft')

powers = np.abs(sft_data[ifo]).astype(np.float64) ** 2
Nfreqs,Nsft = powers.shape

psds = np.zeros((Nfreqs, Nsft))
median_width = 101   # must be odd

for i in range(Nsft):
    psds[:,i] = medfilt(powers[:,i], kernel_size=median_width) / np.log(2)

whitened_power = powers / psds

# plt.hist(whitened_power[:,0],bins=100);
print("means: ",np.mean(whitened_power,axis=0))
print("medians: ",np.median(whitened_power,axis=0))
print("stds: ",np.std(whitened_power,axis=0))


In [ ]:
## Plot one spectrum
plt.figure(figsize=(10, 4))
plt.semilogy(freqs amp[:,0], lw=0.8)
plt.semilogy(freqs, np.sqrt(psds[:,0]))
plt.xlabel("Frequency [Hz]")
plt.ylabel("amplitude")
plt.title(f"{ifo} single SFT spectrum")
plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
#plot a spectrogram

plt.pcolormesh((times-times[0])/86400, freqs, whitened_power, shading="auto")
plt.xlabel("Time")
plt.ylabel("Frequency [Hz]")
plt.colorbar(label="Normalized power")
# plt.ylim([1889.9,1890.1])
plt.tight_layout()